# Supplementary tables from AnnData

This notebook extracts the submission tables from the prepared AnnData bundle. Discovery and validation clinical/classification views are generated from their canonical `.uns` tables. Assay annotations are read from raw-probe metadata. Laboratory documentation and classifier reference material are stored once in `uns["supplementary"]`.

Numerical figure tables are explicitly identified as captured analysis outputs. Their input and producer-code fingerprints must match before export; this notebook does not rerun statistical models. The staining protocol includes antibody identities, dilutions and incubation conditions, without volume amounts. Extraction does not modify AnnData.

Ciavarella groups and spatial archetypes are exported with their canonical source names. No display-time label mappings are applied.


In [1]:
import os
import sys
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
REPO_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
sys.path.insert(0, str(REPO_ROOT))
from dlbcl.supplementary_source import load_source, export_source_tables
from dlbcl.supplementary_xlsx import write_workbook

ADATA_PATH = Path(os.environ.get("DLBCL_DATA_DIR", REPO_ROOT / "data")) / "DLBCL_location_2026.h5ad"
OUTDIR = Path(os.environ.get("DLBCL_SUPPLEMENTARY_OUTDIR", REPO_ROOT / "data/supplementary/submission"))
EXPORT_DATE = datetime.now(ZoneInfo("Europe/Amsterdam")).date().isoformat()
WRITE_XLSX = os.environ.get("DLBCL_EXPORT_XLSX", "1") == "1"


## Validate the data source

Validation rejects a missing supplementary schema, changed scientific inputs, changed analysis-producer code, corrupt result tables and conflicting raw/processed probe annotations. Historical audit records are provenance; they are not alternative sources of current classifier calls.


In [2]:
source, source_sha256 = load_source(ADATA_PATH)
store = source.uns["supplementary"]
catalog = pd.DataFrame(json.loads(store["catalog_json"]))
print("Validated AnnData SHA-256:", source_sha256)
print("Discovery:", len(source.uns["case_classifications"]), "cases")
print("Validation:", len(source.uns["validation_cohort"]["case_classification_validation"]), "cases")
display(catalog[["section", "name", "kind", "canonical_source"]])


Validated AnnData SHA-256: 1ec2d258f6a9f34155657670d5836d4bb058dee9e60f96e7ecef6dcbb141dc0e
Discovery: 64 cases
Validation: 303 cases


,section,name,kind,canonical_source
0,S1,S1 Discovery,view,uns/case_clinical + uns/case_classifications +...
1,S1,S1 Validation,view,uns/validation_cohort/meta + uns/validation_co...
2,S1,S1 Cohort summary,view,"Canonical clinical tables, aggregated by table..."
3,S1,S1 Field definitions,view,uns/supplementary/field_definitions_json + cur...
4,S2,S2 IMC antibody panel,resource,uns/supplementary/tables/imc_antibody_panel/data
...,...,...,...,...
66,S5,S5 RNA coverage,view,Reference memberships + canonical assay annota...
67,S5,S5 DNA features,derived_result,uns/supplementary/tables/table_69/data
68,S5,S5 HMRN LymphPlex features,resource,uns/supplementary/tables/table_70/data
69,S5,S5 Kotlov signatures,view,uns/supplementary/kotlov_signatures + canonica...


## Extract and inspect

The CSVs and workbook specification below are exports, not independent inputs. Re-running extraction replaces them from AnnData. Clinical source/analysis discrepancies are retained and documented rather than adjudicated by the exporter.


In [3]:
tables = export_source_tables(ADATA_PATH, OUTDIR, export_date=EXPORT_DATE)
by_name = {table["name"]: table for table in tables}
for name in ["S2 IMC antibody panel", "S2 Staining protocol", "S4 BLYM777 targets"]:
    table = by_name[name]
    print(name, "—", len(table["rows"]), "rows")
    display(pd.DataFrame(table["rows"], columns=table["headers"]).head(8))


S2 IMC antibody panel — 41 rows


,Target,Clone,Metal,Supplier,Segmentation,T-cell immunity,DC or macrophage,Resident cells,Tumor cells,Recorded intended use,Recorded note
0,SMACT,D4K9N,115In,CST,None,None,None,*,None,"Endothelium, myofibroblast/CAF",None
1,HLA-DR,TAL 1B5,141 Pr,ThermoFisher,*,None,*,None,None,APC's,None
2,CD20,H1,142 Nd,BD Bioscience,*,None,None,None,*,panBcel,None
3,CD68,D4B9C,143 Nd,CST,*,None,*,None,None,macrophage,None
4,CD11b,D6X1N,144 Nd,cst,None,None,*,None,None,macrophage and microglia,None
5,CD4 + 2nd Ab,EPR6855,145 Nd*,Abcam,None,*,None,None,None,T-helper,None
6,CD8a,D8A8Y,146 Nd,CST,None,*,None,None,None,cytotoxic T-cell,None
7,CD31,89C2,147 Sm,CST,None,None,None,None,None,Endothelium,None


S2 Staining protocol — 41 rows


,Step in incubation group,Target or component,Clone,Metal,Supplier,Incubation,Temperature,Dilution denominator (1:x),Recorded performance
0,1.0,CD4 + 2nd Ab,EPR6855,145 Nd*,Abcam,Indirect ON,4C,100.0,medium
1,2.0,TCRgd + 2nd Ab,H41,148 Nd,santa cruz,Indirect ON,4C,50.0,medium
2,1.0,HLA-DR,TAL 1B5,141 Pr,ThermoFisher,5h,RT,100.0,strong
3,2.0,CD11b,D6X1N,144 Nd,cst,5h,RT,100.0,medium
4,3.0,CD8a,D8A8Y,146 Nd,CST,5h,RT,50.0,medium
5,4.0,Granzyme B,D6E9W,150 Nd,cst,5h,RT,50.0,medium
6,5.0,cleaved caspase,5A1E,155 Gd,cst,5h,RT,100.0,dim
7,6.0,VISTA,D1L2G(TM),158 Gd,cst,5h,RT,100.0,medium


S4 BLYM777 targets — 797 rows


,Probe or gene symbol,Transcript accession,Code class
0,ABHD17C,NM_021214.1,Endogenous
1,ACOD1,NM_001258406.1,Endogenous
2,ACTA2,NM_001613.2,Endogenous
3,ACTG2,NM_001615.3,Endogenous
4,ACTL7A,NM_006687.2,Endogenous
5,ADA,NM_000022.2,Endogenous
6,ADGRA2,NM_032777.9,Endogenous
7,ADHFE1,NM_144650.2,Endogenous


## Write the submission workbook

The Excel file uses the same extracted tables as the CSVs, with an index, source descriptions, filterable columns and frozen identifiers. Numeric values retain source precision. Set `DLBCL_EXPORT_XLSX=0` when only the tabular exports and workbook specification are needed.


In [4]:
if WRITE_XLSX:
    workbook = write_workbook(tables, OUTDIR / "Supplementary_Tables_Cancer_Discovery.xlsx")
    print("Wrote", workbook.name)
else:
    print("Extracted", len(tables), "tables; workbook_data.json is ready for rendering.")


Extracted 73 tables; workbook_data.json is ready for rendering.
